In [1]:

import time
from typing import Any

import gymnasium as gym
import numpy as np
import torch
from torch import optim, nn

from src.experiment_logging.experiment_log import ExperimentLogItem
from src.experiment_logging.experiment_logger import ExperimentLogger, log_experiment
from src.module_analysis import count_parameters
from src.reinforcement_learning.algorithms.sac.sac import SAC, SACInfoStashConfig
from src.reinforcement_learning.algorithms.sac.sac_policy import SACPolicy
from src.reinforcement_learning.core.action_selectors.predicted_std_action_selector import PredictedStdActionSelector
from src.reinforcement_learning.core.callback import Callback
from src.reinforcement_learning.core.loss_config import LossInfoStashConfig
from src.reinforcement_learning.core.policies.components.actor import Actor
from src.reinforcement_learning.core.policies.components.q_critic import QCritic
from src.reinforcement_learning.gym.parallelize_env import parallelize_env_async
from src.stopwatch import Stopwatch
from src.summary_statistics import maybe_compute_summary_statistics
from swarmbots.envs.swarm_bot_env__v0_1 import SwarmBotEnv

%load_ext autoreload
%autoreload 2

pygame 2.5.2 (SDL 2.28.3, Python 3.11.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
device = torch.device("cuda:0") if True else torch.device('cpu')
print(f'using device {device}')

env_kwargs = {}
num_envs = 1

def create_env(render_mode: str | None):
    make_single_env = lambda: SwarmBotEnv(
        physics_steps_per_step=10, 
        action_scale=5, 
        hip_range=np.pi / 4,
        ctrl_cost_weight=0.001,
    )
    
    if num_envs == 1:
        return make_single_env()
        
    return parallelize_env_async(make_single_env, num_envs)


def create_policy():
    in_size = int(np.prod(env.observation_space.shape))
    latent_action_size = 256
    action_size = int(np.prod(env.action_space.shape))
    
    actor_net = nn.Sequential(
        nn.Linear(in_size, 256),
        nn.ReLU(),
        nn.Linear(256, latent_action_size),
        nn.ReLU(),
    )

    critic = QCritic(
        n_critics=2,
        create_q_network=lambda: nn.Sequential(
            nn.Linear(in_size + action_size, 256),
            nn.ReLU(),
            # BatchRenorm(256),
            nn.Linear(256, 256),
            nn.ReLU(),
            # BatchRenorm(256),
            nn.Linear(256, 1)
        )
    )

    return SACPolicy(
        actor=Actor(actor_net, PredictedStdActionSelector(
            latent_dim=latent_action_size,
            action_dim=action_size,
            base_std=1.0,
            squash_output=True,
        )),
        critic=critic
    )


env = create_env(render_mode=None)
policy = create_policy()

using device cuda:0


In [5]:
policy_id = '2024-11-02_17-43-07_394325~911Dc5_best'
policy.load_state_dict(torch.load(f'saved_models/{policy_id}.pth'))

<All keys matched successfully>

In [6]:
from src.reinforcement_learning.core.policy_evaluation import record_policy

env.render_mode = 'rgb_array'
record_policy(
    env=env,
    policy=policy,
    video_folder='videos',
    video_name_prefix=policy_id,
    deterministic_actions=False,
    num_steps=999,
    torch_device=device,
)

Moviepy - Building video C:\Users\domin\Git\swarm-bots\videos\2024-11-02_17-43-07_394325~911Dc5_best-episode-0.mp4.
Moviepy - Writing video C:\Users\domin\Git\swarm-bots\videos\2024-11-02_17-43-07_394325~911Dc5_best-episode-0.mp4


Moviepy - Done !
Moviepy - video ready C:\Users\domin\Git\swarm-bots\videos\2024-11-02_17-43-07_394325~911Dc5_best-episode-0.mp4
closing record env
Moviepy - Building video C:\Users\domin\Git\swarm-bots\videos\2024-11-02_17-43-07_394325~911Dc5_best-episode-1.mp4.
Moviepy - Writing video C:\Users\domin\Git\swarm-bots\videos\2024-11-02_17-43-07_394325~911Dc5_best-episode-1.mp4


Moviepy - Done !
Moviepy - video ready C:\Users\domin\Git\swarm-bots\videos\2024-11-02_17-43-07_394325~911Dc5_best-episode-1.mp4
record env closed


In [10]:
env.physics.data.xpos

array([[ 0.        ,  0.        ,  0.        ],
       [10.55573153, -0.4078065 ,  0.14921404],
       [10.55573153, -0.4078065 ,  0.14921404],
       [10.48739983, -0.43612304,  0.21651153],
       [10.48739983, -0.43612304,  0.21651153],
       [10.48739983, -0.43612304,  0.21651153],
       [10.64728733, -0.4268981 ,  0.18461243],
       [10.64728733, -0.4268981 ,  0.18461243],
       [10.64728733, -0.4268981 ,  0.18461243],
       [10.54656564, -0.30979478,  0.13161603],
       [10.54656564, -0.30979478,  0.13161603],
       [10.54656564, -0.30979478,  0.13161603],
       [10.02058614, -0.30683126,  0.33239675],
       [10.02058614, -0.30683126,  0.33239675],
       [10.11026708, -0.30289121,  0.37646266],
       [10.11026708, -0.30289121,  0.37646266],
       [10.11026708, -0.30289121,  0.37646266],
       [10.00786531, -0.22566631,  0.27538418],
       [10.00786531, -0.22566631,  0.27538418],
       [10.00786531, -0.22566631,  0.27538418],
       [ 9.94918638, -0.30995894,  0.402

In [2]:
import PIL.Image

PIL.Image.fromarray(create_env(None).render())

NameError: name 'create_env' is not defined